# 1주차: 나나를 깨우다
## Nana 개인 일정 tool

### 학습 목표
- `personal_create_schedule`, `personal_list_schedules`, `personal_delete_schedule`의 역할을 구분한다.
- 모델이 개인 일정 tool을 어떤 arguments로 호출하는지 trace로 확인한다.
- 메모리 저장소에 내 개인 일정이 생성, 조회, 삭제되는 흐름을 설명한다.

### 핵심 개념
Week 1은 Nana가 내 개인 일정을 다루는 가장 작은 agentic 흐름이다.
모델은 일정을 직접 저장하지 않고, 개인 일정 tool 이름과 arguments를 만든다.

## 환경 설정

In [ ]:
import json
import sys
sys.path.insert(0, '..')

from student_parts.week01_wake_up_nana import (
    personal_create_schedule,
    personal_list_schedules,
    personal_delete_schedule,
    PERSONAL_SCHEDULES,
)

# 기존 일정 초기화
PERSONAL_SCHEDULES.clear()
print("환경 설정 완료")

## 1. 참석자가 없는 개인 일정 생성

In [ ]:
# 참석자가 없는 개인 일정 생성
result_no_attendees = personal_create_schedule.invoke({
    "title": "개인 독서 시간",
    "date": "2026-07-10",
    "start_time": "09:00",
    "end_time": "10:00",
})
trace_no_attendees = json.loads(result_no_attendees)
print("=== 참석자 없는 일정 생성 결과 ===")
print(json.dumps(trace_no_attendees, indent=2, ensure_ascii=False))

## 2. 참석자가 있는 개인 일정 생성

In [ ]:
# 참석자가 있는 개인 일정 생성
result_with_attendees = personal_create_schedule.invoke({
    "title": "팀 점심 미팅",
    "date": "2026-07-11",
    "start_time": "12:00",
    "end_time": "13:00",
    "attendees": ["철수", "영희"],
})
trace_with_attendees = json.loads(result_with_attendees)
print("=== 참석자 있는 일정 생성 결과 ===")
print(json.dumps(trace_with_attendees, indent=2, ensure_ascii=False))

## 3. 두 trace 비교: attendees 필드 차이

In [ ]:
# attendees 필드 비교
print("참석자 없는 일정의 attendees:", trace_no_attendees["created_schedule"]["attendees"])
print("참석자 있는 일정의 attendees:", trace_with_attendees["created_schedule"]["attendees"])
print()
print("→ attendees가 None이면 빈 list[]로 변환되고, 값이 있으면 그대로 저장됩니다.")
print()

# 전체 payload 구조 비교
print("=== payload 키 비교 ===")
s1 = trace_no_attendees["created_schedule"]
s2 = trace_with_attendees["created_schedule"]
for key in s1:
    v1 = s1[key]
    v2 = s2[key]
    marker = "  ← 차이" if v1 != v2 else ""
    print(f"  {key:15s} | {str(v1):25s} | {str(v2):25s}{marker}")

## 4. 일정 조회 (personal_list_schedules)

In [ ]:
# 현재 일정 목록 조회
result_list = personal_list_schedules.invoke({})
trace_list = json.loads(result_list)
print("=== 전체 일정 조회 ===")
print(json.dumps(trace_list, indent=2, ensure_ascii=False))
print(f"\n총 {len(trace_list['schedules'])}건의 일정이 조회되었습니다.")

## 5. 일정 삭제 (personal_delete_schedule)

In [ ]:
# 첫 번째 일정(참석자 없는 일정) 삭제
first_id = trace_no_attendees["created_schedule"]["id"]
print(f"삭제할 일정 ID: {first_id}")
print()

result_delete = personal_delete_schedule.invoke({"schedule_id": first_id})
trace_delete = json.loads(result_delete)
print("=== 일정 삭제 결과 ===")
print(json.dumps(trace_delete, indent=2, ensure_ascii=False))

In [ ]:
# 삭제 후 조회
result_after = personal_list_schedules.invoke({})
trace_after = json.loads(result_after)
print("=== 삭제 후 일정 목록 ===")
print(json.dumps(trace_after, indent=2, ensure_ascii=False))
print(f"\n남은 일정: {len(trace_after['schedules'])}건")

## 관찰 요약

| 항목 | 참석자 없는 일정 | 참석자 있는 일정 |
|------|-----------------|----------------|
| attendees | `[]` (빈 list) | `["철수", "영희"]` |
| id | personal_XXXX | personal_XXXX |
| tool_name | personal_create_schedule | personal_create_schedule |

### 확인 질문 답변
1. **모델 답변과 tool call의 차이**: 모델 답변은 자연어 텍스트이고, tool call은 함수명 + arguments의 구조화된 요청이다.
2. **사람이 검증해야 할 arguments**: `date`, `start_time`, `end_time`은 사용자 의도와 다를 수 있으므로 반드시 확인해야 한다.
3. **저장소 payload 확인 이유**: 모델이 자연스럽게 답변해도, 실제 저장된 데이터가 의도와 다를 수 있다 (예: 날짜 오류, 참석자 누락).